# 01 — EDA & Feature Engineering
**Project:** Detecting anomalies in network traffic (UNSW-NB15)

This notebook covers Step 3 of the ML lifecycle: data cleaning, applied EDA, feature engineering, explainability (SHAP), feature selection, and dimensionality reduction (PCA + t-SNE). All transformations are **fit on the training split only** to prevent information leakage — a 2026 NIDS best practice for non-stationary traffic.

In [ ]:
import sys, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, 'src')
import preprocessing as pp
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
ACCENT = '#1F4E79'

## 1. Load data
We use the official UNSW-NB15 train/test partition (175,341 / 82,332 records).

In [ ]:
train = pp.load_raw('data/UNSW_NB15_training-set.csv')
test  = pp.load_raw('data/UNSW_NB15_testing-set.csv')
print(train.shape, test.shape)
train.head()

## 2. Data quality
No nulls or duplicates, but `service` uses `'-'` as an encoded placeholder (~54% of rows) which we treat as its own category.

In [ ]:
print('missing:', int(train.isnull().sum().sum()))
print('duplicates:', int(train.duplicated().sum()))
print("service == '-':", round((train['service']=='-').mean()*100, 1), '%')
print('\nbinary balance:\n', train['label'].value_counts(normalize=True).round(3))
print('\nattack categories:\n', train['attack_cat'].value_counts())

## 3. Applied EDA
Class balance, attack-family breakdown, heavy right-skew of volume features (log-compression fixes it), and attack rate by service/state.

In [ ]:
# see figures/eda_class_balance.png, eda_distributions.png, eda_attack_rate_by_cat.png
key = ['sbytes','dbytes','sload','dur']
print('skew before log:', {c: round(train[c].skew(),1) for c in key})
print('skew after  log:', {c: round(np.log1p(train[c].clip(lower=0)).skew(),1) for c in key})

## 4. Feature engineering & preprocessing
Pipeline (`src/preprocessing.py`): drop `id`; `service='-'`→`none`; **domain features** (bytes-per-packet, source/dest byte ratio, totals, packet ratio, TTL difference); one-hot encode `service`/`state`; frequency-encode high-cardinality `proto`; log-compress skewed features; **RobustScaler** (median/IQR, outlier-resistant). Encoders and scaler are fit on train only.

In [ ]:
art = pp.fit_preprocessor(train)
Xtr, ytr, ytr_m = pp.transform(train, art)
Xte, yte, yte_m = pp.transform(test, art)
print('features after engineering:', Xtr.shape[1])
print('domain features added:', ['bytes_per_pkt_src','bytes_per_pkt_dst',
      'src_dst_byte_ratio','total_bytes','total_pkts','pkt_ratio','ttl_diff'])

## 5. Explainability — Random Forest importance + SHAP
A Random Forest ranks feature importance; SHAP (TreeExplainer) confirms which features drive attack predictions and in which direction. TTL-based features (`sttl`, `ttl_diff`, `ct_state_ttl`) dominate — consistent with intrusion-detection theory, where spoofed/crafted packets show abnormal TTLs. Two engineered features (`ttl_diff`, `src_dst_byte_ratio`) rank in the top five.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import shap
rf = RandomForestClassifier(n_estimators=150, max_depth=16, n_jobs=-1,
                            class_weight='balanced', random_state=42).fit(Xtr, ytr)
imp = pd.Series(rf.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print(imp.head(10))
# SHAP on a sample (see figures/fe_shap_summary.png)
samp = Xtr.sample(800, random_state=42)
sv = shap.TreeExplainer(rf).shap_values(samp, check_additivity=False)
sv_pos = sv[...,1] if getattr(sv,'ndim',2)==3 else (sv[1] if isinstance(sv,list) else sv)
shap.summary_plot(sv_pos, samp, max_display=12)

## 6. Feature selection (embedded + filter)
Embedded: keep the top-30 features by RF importance. Filter: drop the lower-importance partner of any pair with |correlation| > 0.9. Result: a compact ~21-feature set retained for interpretable/lightweight models.

In [ ]:
sel = json.load(open('models/feature_selection.json'))['selected_features']
print(len(sel), 'features selected:\n', sel)

## 7. Dimensionality reduction — PCA & t-SNE
For PCA/t-SNE we z-score standardize first so no single feature dominates the components. ~31 of 69 components retain 95% variance. t-SNE shows normal and attack traffic form largely separable clusters (see figures/fe_pca_scree.png, fe_tsne.png).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Xz = StandardScaler().fit_transform(Xtr)
pca = PCA(n_components=0.95, random_state=42).fit(Xz)
print(f'{pca.n_components_} / {Xtr.shape[1]} components explain 95% variance')

In [ ]:
# persist processed splits for the modelling notebook
Xtr.assign(label=ytr, attack_cat=ytr_m).to_parquet('data/train_processed.parquet')
Xte.assign(label=yte, attack_cat=yte_m).to_parquet('data/test_processed.parquet')
import joblib; joblib.dump(art, 'models/preprocessor.joblib')
print('saved processed data + preprocessor')